# E-field → Voltage with Galactic Noise — Demo

This notebook demonstrates the full pipeline:
1. Load the RF-chain configuration from a JSON file.
2. Build the transfer function and load the antenna effective lengths.
3. Open an E-field ROOT file and read one event.
4. Apply the instrument response to obtain voltage traces.
5. Compute galactic noise and add it to the voltage traces.
6. Plot the results.

## 1. Imports and configuration

In [ ]:
import json
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt

from make_input import load_input_params_from_dict
from apply_rfchain import (
    open_event_root,
    percieved_theta_phi,
    make_full_response_matrix,
    efield_2_voltage,
    voltage_to_adc,
)
from noise import compute_noise

## 2. Load the RF-chain parameters

The JSON file describes every component of the analogue chain (balun, matching network, LNA, cable, VGA, …) and points to the S-parameter files and effective-length maps.

In [ ]:
config_file = "antenna_configs/RF_params_new_leffs.json"

with open(config_file, "r") as f:
    params_RF = json.load(f)

# load_input_params_from_dict builds the transfer function from S-parameters
# and loads the effective-length maps for the three antenna arms (SN, EW, Z).
(
    duration, latitude, altitude,
    input_sampling_freq, out_sampling_freq,
    N_samples, sampling_period, freqs,
    out_N_samples, out_sampling_period, out_freqs,
    LST_radians, tf, t_SN, t_EW, t_Z,
) = load_input_params_from_dict(params_RF)

print(f"Duration            : {duration*1e6:.3f} µs")
print(f"Input sampling freq : {input_sampling_freq/1e9:.1f} GHz  →  {N_samples} samples")
print(f"Output sampling freq: {out_sampling_freq/1e6:.0f} MHz  →  {out_N_samples} samples")
print(f"Detector latitude   : {latitude*180/np.pi:.2f}° (co-latitude)")
print(f"Transfer function shape (3 arms × n_freq): {tf.shape}")

## 3. Open the E-field ROOT file

Point `root_dir` to a directory containing the standard GRAND ROOT files:
- `run_*_L0_*.root` (antenna positions)
- `shower_*_L0_*.root` (shower metadata)
- `efield_*_L0_*.root` (E-field traces)

In [ ]:
# ── Point this to your ROOT directory ──
root_dir = "EXPLORATION/nonoise_00"

# Read the first event only (start=0, stop=1)
antenna_pos, meta_data, efield_data = open_event_root(root_dir, start=0, stop=1)

print(f"Number of events loaded  : {len(efield_data['traces'])}")
print(f"Antennas in first event  : {efield_data['traces'][0].shape[0]}")
print(f"Trace length per antenna : {efield_data['traces'][0].shape[-1]} samples")
print(f"Shower energy            : {meta_data['energy_primary'][0]:.2e} eV")
print(f"Zenith / Azimuth         : {np.degrees(meta_data['zenith'][0]):.1f}° / {np.degrees(meta_data['azimuth'][0]):.1f}°")

## 4. Convert E-field to voltage (clean, no noise)

Steps:
1. FFT the E-field traces.
2. Compute the perceived direction (θ, φ) from every antenna to Xmax.
3. Build the full response matrix = effective length × transfer function.
4. Multiply in the frequency domain and inverse-FFT to get voltage traces.

In [ ]:
ev = 0  # event index

# E-field traces → FFT
event_traces = efield_data["traces"][ev].astype(np.float64)
event_trace_fft = sp.fft.rfft(event_traces)

# Antenna positions and Xmax for this event
ant_pos = antenna_pos[efield_data["du_id"][ev]]
xmax_pos = meta_data["xmax_pos"][ev]

# Perceived direction from each antenna to Xmax
theta_du, phi_du = percieved_theta_phi(ant_pos, xmax_pos)

# Full response matrix: (n_ant, 3_Efield_polar, 3_channels, n_freq)
full_response = make_full_response_matrix(
    t_SN, t_EW, t_Z, theta_du, phi_du, tf,
    duration=duration, input_sampling_freq=input_sampling_freq,
)

# E-field × response → voltage
vout_clean, vout_clean_f = efield_2_voltage(
    event_trace_fft, full_response,
    current_rate=input_sampling_freq, target_rate=input_sampling_freq,
)

print(f"Voltage output shape (n_ant, 3_channels, n_samples): {vout_clean.shape}")

## 5. Compute galactic noise and generate noise traces

The `compute_noise` class integrates the sky brightness temperature (from LFmap) weighted by the antenna effective area over the visible sky, for each Local Sidereal Time (LST).

This gives the noise power spectral density $P_\nu(\text{LST})$, which is then used to draw Gaussian realisations in the frequency domain.

In [ ]:
# Initialise the noise computer
# lst_time_resolution=10 h means we sample at 0, 10, 20 h LST (fast for demo)
noise_computer = compute_noise(
    lst_time_resolution=10.0,
    detector_lat=latitude,
    list_temp_files=[f"files/LFmap/LFmapshort{i}.npy" for i in range(20, 251)],
    LF_freqs=np.arange(20, 251) * 1e6,
    target_freqs=out_freqs,
    tf_rfchain=tf,
    duration=duration,
    leff_x=t_SN,
    leff_y=t_EW,
    leff_z=t_Z,
)

print("Noise computer initialised.")
print(f"LST grid: {noise_computer.lst_hours} hours")
print(f"LFmap frequency range: {noise_computer.LF_freqs.min()/1e6:.0f} – {noise_computer.LF_freqs.max()/1e6:.0f} MHz")

In [ ]:
# Choose an LST hour and generate noise samples (one per antenna)
lst_hour = 6.0  # hours
n_antennas = vout_clean.shape[0]

noise_traces, noise_fft = noise_computer.noise_samples(
    lst_hour=lst_hour, n_samples=n_antennas, seed=42, micro=True,
)
# noise_traces shape: (n_antennas, 3_channels, n_time_samples) in µV
print(f"Noise traces shape: {noise_traces.shape}")
print(f"Noise RMS per channel [µV]: {noise_traces.std(axis=(0, 2))}")

## 6. Add noise to clean voltage and plot

The clean voltage `vout_clean` is in µV (after ×1e6 scaling). The noise traces from `noise_samples(..., micro=True)` are also in µV, so we can add them directly.

**Note:** the clean trace has the input sampling rate (2 GHz), while the noise is generated at the output rate (500 MHz). We downsample the clean trace first.

In [ ]:
# Downsample clean voltage from input rate to output rate
downsample_factor = int(input_sampling_freq / out_sampling_freq)
vout_clean_ds = vout_clean[..., ::downsample_factor] * 1e6  # → µV

# Match lengths (noise may differ by 1 sample)
n_min = min(vout_clean_ds.shape[-1], noise_traces.shape[-1])
vout_clean_ds = vout_clean_ds[..., :n_min]
noise_traces  = noise_traces[..., :n_min]

# Add noise
vout_noisy = vout_clean_ds + noise_traces
print(f"Clean voltage shape  : {vout_clean_ds.shape}")
print(f"Noisy voltage shape  : {vout_noisy.shape}")

In [ ]:
# ── Plot one antenna ──
ant_idx = 0
times_us = np.arange(n_min) / out_sampling_freq * 1e6  # µs
labels = ["SN (X)", "EW (Y)", "Z"]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
for ch in range(3):
    axes[ch].plot(times_us, vout_noisy[ant_idx, ch], alpha=0.7, color=colors[ch], label="Signal + Noise")
    axes[ch].plot(times_us, vout_clean_ds[ant_idx, ch], color="k", lw=1.2, label="Clean signal")
    axes[ch].set_ylabel(f"{labels[ch]} [µV]")
    axes[ch].legend(loc="upper right", fontsize=9)
    axes[ch].grid(True, alpha=0.3)
axes[-1].set_xlabel("Time [µs]")
axes[0].set_title(
    f"Voltage trace — antenna {ant_idx}  |  LST = {lst_hour:.0f} h  |  "
    f"E = {meta_data['energy_primary'][ev]:.1e} eV  |  "
    f"θ = {np.degrees(meta_data['zenith'][ev]):.1f}°"
)
plt.tight_layout()
plt.show()

In [ ]:
# ── PSD comparison: clean vs noisy ──
psd_clean = np.abs(sp.fft.rfft(vout_clean_ds[ant_idx]))**2 / (n_min * out_sampling_freq) * 1e6  # µV²/MHz
psd_noisy = np.abs(sp.fft.rfft(vout_noisy[ant_idx]))**2 / (n_min * out_sampling_freq) * 1e6
psd_freqs = sp.fft.rfftfreq(n_min, 1 / out_sampling_freq) / 1e6  # MHz

fig, ax = plt.subplots(figsize=(10, 5))
for ch in range(3):
    ax.plot(psd_freqs, psd_noisy[ch], alpha=0.7, color=colors[ch], label=f"{labels[ch]} noisy")
    ax.plot(psd_freqs, psd_clean[ch], ls="--", color=colors[ch], label=f"{labels[ch]} clean")
ax.set_yscale("log")
ax.set_xlabel("Frequency [MHz]")
ax.set_ylabel("PSD [µV²/MHz]")
ax.set_title(f"Power spectral density — antenna {ant_idx}")
ax.set_xlim(20, 250)
ax.legend(ncol=2, fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()